<a href="https://colab.research.google.com/github/serenamom/ICT302-Sample-Project/blob/main/Advanced_Data_Analytics_Group_Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import io
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from google.colab import files
from IPython.display import display

**Step 1: Upload Train dataset**

In [ ]:
#Upload CSV file.
uploaded = files.upload()

In [ ]:
#Get the uploaded filename
filename = list(uploaded.keys())[0]

In [ ]:
#read dataset
df = pd.read_csv(filename, sep = None, engine = "python")

In [ ]:
#remove spaces around column names
df.columns = df.columns.str.strip()

In [ ]:
#blank cells as missing values
df = df.replace(r"^|s*$", np.nan, regex=True)

In [ ]:
#display the first five rows
display(df.head())

In [ ]:
print(df.shape)

In [ ]:
print(df.columns.tolist())

In [ ]:
print(df.isna().sum())

In [ ]:
print(df.duplicated().sum())

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(filename)

In [ ]:
df.columns = df.columns.str.strip()

**Step 2: Check and Clean the Data**

In [ ]:
#check that the required columns exist
required_columns = ["date", "store", "item", "sales"]
if not set(required_columns).issubset(df.columns):
  raise ValueError("The dataset must contain date, store, item, and sales.")

In [ ]:
#convert the date column into actual dates
df["date"] = pd.to_datetime(df["date"], errors="coerce")

In [ ]:
#check whether any dates could not be converted
print("Missing or invalid dates:", df["date"].isna().sum())

In [ ]:
display(df.head())

In [ ]:
#keep a separate copy
data = df[required_columns].copy()

In [ ]:
#convert blank text into missing values
data = data.replace(r"^|s*$", np.nan, regex = True)

In [ ]:
#convert dates and sales into appropriate formats
data["date"] = pd.to_datetime(data["date"], errors = "coerce")
data["sales"] = pd.to_numeric(data["sales"], errors = "raise")

In [ ]:
print("Missing values:")
print(data.isna().sum())
print("\nExact duplicate rows:", data.duplicated().sum())

In [ ]:
#stop if important identifiers or dates are missing
if data[["date", "store", "item"]].isna().any().any():
  raise ValueError(
      "Investigate missing dates, stores, or items before continuing."
  )

In [ ]:
#a training record needs a known sales amount
data = data.dropna(subset=["sales"]).copy()

In [ ]:
#stop if unexpected negatve sales are present
if(data["sales"] < 0).any():
  raise ValueError(
      "Negative sales found. check whether these represent returns."
  )

In [ ]:
#check that each store item and date has one daily sales record
if data.duplicated(["date", "store", "item"]).any():
  raise ValueError(
      "Repeated store-item-date records found."
      "Investigate whether to remove duplicates or aggregate transactions."
  )

In [ ]:
#arrange records from oldest to newest
data = data.sort_values(
    ["date", "store", "item"]
).reset_index(drop=True)
print("\nCleaned shape:", data.shape)
print("Earliest date:", data["date"].min())
print("Latest date:", data["date"].max())
print("Stores:", data["store"].nunique())
print("Items:", data["item"].nunique())
display(data.head())

**Step 3: Explore sales patterns**

In [ ]:
#summarise sales per store-item-day
display(data["sales"].describe())

In [ ]:
#add together sales across all stores and items for each date
daily_sales = data.groupby("date")["sales"].sum()
plt.figure(figsize=(12, 4))
daily_sales.plot()
plt.title("Total Daily Sales Across All Stores and Items")
plt.xlabel("Date")
plt.ylabel("Total sales units")
plt.tight_layout()
plt.show()

In [ ]:
#show the distribution of individual daily sales records
plt.figure(figsize=(8, 4))
data["sales"].hist(bins=40)
plt.title("Distribution of Sales per Store-Item_Day")
plt.xlabel("Sales units")
plt.ylabel("Numer of records")
plt.tight_layout()
plt.show()

**Step 4: Create Prediction Features**

In [ ]:
#extract calendar information
data["year"] = data["date"].dt.year
data["month"] = data["date"].dt.month
data["day_of_week"] = data["date"].dt.dayofweek

In [ ]:
#monday = 0 and sunday = 6
data["is weekend"] = (
    data["day_of_week"] >= 5
).astype(int)

In [ ]:
#treat identifiers and seasonal calendar values as categories
categorical_features = [
    "store", "item","year", "month", "day_of_week"
]
for column in categorical_features:
  data[column] = data[column].astype(str)

In [ ]:
#keep these as numerical features (removed duplicate 'year')
numerical_features = ["is weekend"]

In [ ]:
# Clean features list definition (ensuring unique columns)
features = categorical_features + numerical_features
target = "sales"
task = "regression"

print("Task:", task)
print("Target:", target)
print("Features:", features)
display(data[["date"] + features + [target]].head())

**Step 5: Split By Date**

In [ ]:
#defining the forecasting horizon
forecast_days = 90
last_date = data["date"].max()
test_start = last_date - pd.Timedelta(
    days=forecast_days - 1
)
validation_start = test_start - pd.Timedelta(
    days=forecast_days
)

In [ ]:
#split complete dates, keeping all atores and items together
train = data[data["date"] < validation_start].copy()
validation = data[
    (data["date"] >= validation_start)
    & (data["date"] < test_start)
].copy()
test = data[data["date"] >= test_start].copy()
if min(len(train), len(validation), len(test)) == 0:
  raise ValueError(
      "Not enough history, choose a shorter forecast_days value."
  )

In [ ]:
# separate prediction inputs from actual sales using corrected features
X_train = train[features]
y_train = train[target]
X_validation = validation[features]
y_validation = validation[target]
X_test = test[features]
y_test = test[target]

In [ ]:
#display the periods used
split_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": len(part),
        "Start": part["date"].min().date(),
        "End": part["date"].max().date(),
    }
    for name, part in [
        ("Train", train),
        ("Validation", validation),
        ("Test", test),
    ]
])
display(split_summary)

**Step 6: Prepare features and define models**

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

In [ ]:
#convert categories into numerical indicator columns
#scale numerical features for the linear model
preparation = ColumnTransformer([
  ("categories",
    OneHotEncoder(handle_unknown="ignore"),
    categorical_features
   ),
  (
      "numbers",
      StandardScaler(),
      numerical_features
  )
])

In [ ]:
#define the approaches to compare
models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=50,
        max_depth=14,
        min_samples_leaf=10,
        max_samples=0.5,
        random_state=42,
        n_jobs=-1
    )
}

**Step 7: Train and compare on validation data**

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

#reusable function for regression evaluation.
def evaluate(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted)
    }

pipelines = {}
validation_results = []

for name, model in models.items():

    print("Training:", name)
    start_time = time.perf_counter()

    pipeline = Pipeline([
        ("preparation", clone(preparation)),
        ("model", clone(model))
    ])

    #learn using training data only.
    pipeline.fit(X_train, y_train)

    #predict the later validation period.
    predicted = pipeline.predict(X_validation)

    #apply the same nonnegative rule to every model.
    predicted = np.maximum(predicted, 0)

    scores = evaluate(y_validation, predicted)

    validation_results.append({
        "Model": name,
        **scores,
        "Fit and predict seconds": (
            time.perf_counter() - start_time
        )
    })

    pipelines[name] = pipeline

#select using the lowest validation MAE.
comparison = pd.DataFrame(validation_results)
comparison = comparison.sort_values("MAE").reset_index(drop=True)

display(comparison.round(3))

best_name = comparison.loc[0, "Model"]

print("Selected model:", best_name)

**Step 8: Evaluate the selected model on the test period**

In [ ]:
#combine all records before the test period
development = pd.concat(
    [train, validation],
    ignore_index=True
)
#create a fresh copy of the selected pipeline
final_model = clone(pipelines[best_name])

#refit using the training and validation records
final_model.fit(
    development[features],
    development[target]
)

#forecast the final test period
test_predictions = final_model.predict(X_test)
test_predictions = np.maximum(test_predictions, 0)

#calculate final test metrics
test_scores = evaluate(y_test, test_predictions)

test_results = pd.DataFrame([{
    "Model": best_name,
    **test_scores
}])

print("Final test performance:")
display(test_results.round(3))

#keep prediction alongside their actual records
prediction_table = test[
    ["date", "store", "item", "sales"]
].copy()

prediction_table["predicted_sales"] = test_predictions

prediction_table["absolute_error"] = (
    prediction_table["sales"] -
    prediction_table["predicted_sales"]
).abs()

display(prediction_table.head(10))

In [ ]:
#compare actual and predicted total saled by date
daily_comparison = prediction_table.groupby("date")[
    ["sales", "predicted_sales"]
].sum()
daily_comparison.plot(figsize=(12, 4))
plt.title("Actual versus Predicted Total Daily Sales")
plt.xlabel("Date")
plt.ylabel("Total sales units")
plt.legend(["Actual sales", "Predicted sales"])
plt.tight_layout()
plt.show()

# Inspect a manageable sample of individual predictions.
sample = prediction_table.sample(
    n=min(3000, len(prediction_table)),
    random_state=42
)

plt.figure(figsize=(6, 6))
plt.scatter(
    sample["sales"],
    sample["predicted_sales"],
    alpha=0.25
)

upper = max(
    sample["sales"].max(),
    sample["predicted_sales"].max()
)

plt.plot([0, upper], [0, upper], "r--")
plt.title("Actual versus Predicted Sales per Record")
plt.xlabel("Actual sales")
plt.ylabel("Predicted sales")
plt.tight_layout()
plt.show()

# Check whether errors vary between stores.
store_errors = prediction_table.groupby("store")[
    "absolute_error"
].mean().sort_values()

store_errors.plot.bar(figsize=(9, 4))
plt.title("Mean Absolute Error by Store")
plt.xlabel("Store")
plt.ylabel("MAE in sales units")
plt.tight_layout()
plt.show()

In [ ]:
# Save the tables as CSV files.
comparison.to_csv("validation_comparison.csv", index=False)
test_results.to_csv("final_test_results.csv", index=False)
prediction_table.to_csv("sales_predictions.csv", index=False)

# Download each file.
files.download("validation_comparison.csv")
files.download("final_test_results.csv")
files.download("sales_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>